## Reranker Hybrid Search Algorithm

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma,FAISS


#utility
import numpy as np
from typing import List,Dict,Any
from sentence_transformers import SentenceTransformer

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import (RunnablePassthrough,RunnableMap)
from langchain_core.output_parsers import StrOutputParser
import os
import numpy as np
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [2]:
##load text files
loader=TextLoader("langchain_sample.txt")
raw_docs=loader.load()

splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
docs=splitter.split_documents(raw_docs)
docs

[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.\nBM25 is a traditional 

In [3]:
##user query
query="How can i use langchain to build an application with RAG architecture?"

In [5]:
## Vector store faiss
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(docs,embedding_model)
retriever=vectorstore.as_retriever(search_kwargs={"k":8})


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"

)
vectorstore_diff=FAISS.from_documents(docs,embeddings)
retriever_diff=vectorstore_diff.as_retriever(search_kwargs={"k":8})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001FF8947E710>, search_kwargs={'k': 8})

In [8]:
retriever_diff

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001FF882A1E50>, search_kwargs={'k': 8})

In [ ]:
## Prompt and use LLM
llm=init_chat_model(model="groq:llama-3.3-70b-versatile",api_key="")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020031D22B50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000200328350D0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [22]:
prompt=PromptTemplate.from_template("""
You are a helpful asisstant.Your task is to rank the following documents from most to least relevant.
User Question:"{question}"
Documents:
 {documents}    
Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order ,starting from the most relevant.

Output format :comma-seperated document indeices (e.g. ,2,1,3,0,.....)                                
""")



In [23]:
retrieverd_docs=retriever.invoke(query)
retrieverd_docs

[Document(id='2d74bfcb-64fe-42d9-97ca-5335386fee56', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='d5ba7c11-4ddb-407c-a952-a4dcc3aae5f5', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(id='5915d59b-5712-4eb0-9892-6eaea0dc28da', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond mo

In [24]:
chain=prompt|llm|StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful asisstant.Your task is to rank the following documents from most to least relevant.\nUser Question:"{question}"\nDocuments:\n {documents}    \nInstructions:\n- Think about the relevance of each document to the user\'s question.\n- Return a list of document indices in ranked order ,starting from the most relevant.\n\nOutput format :comma-seperated document indeices (e.g. ,2,1,3,0,.....)                                \n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': Fals

In [25]:
doc_lines=[f"{i+1}.{doc.page_content}" for i,doc in enumerate(retrieverd_docs)]
formatted_docs="\n".join(doc_lines)

In [26]:
response=chain.invoke({"question":query,"documents":formatted_docs})

In [27]:
doc_lines

['1.LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '2.LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '3.LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.',
 '4.Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM

In [28]:
response

'To answer the user\'s question, "How can I use LangChain to build an application with RAG architecture?", we need to identify the documents that provide the most relevant information about using LangChain for building applications, specifically with RAG (Retrieval-Augmented Generation) architecture.\n\n1. **Document 4** is the most relevant because it directly mentions RAG and how LangChain supports its implementation using vector databases like FAISS, Chroma, and Pinecone. This directly addresses the user\'s question about using LangChain with RAG architecture.\n\n2. **Document 1** provides a general overview of LangChain, including its purpose and components. While it doesn\'t specifically mention RAG, it\'s foundational information that would be useful for someone looking to build an application with LangChain.\n\n3. **Document 2** talks about LangChain\'s integration with third-party services, which could be useful for optimizing performance but doesn\'t directly relate to RAG or 

In [29]:
indices=[int(x.strip())-1 for x in response.split(",")  if x.strip().isdigit()]
indices

[0, 2, 1, 5, 0, 2, 1, 5, 4]